# Advanced 01 — 3D Vision & Spatial Intelligence

## From camera geometry to metric scene understanding

**Scenario.** A calibrated stereo rig must measure a valve assembly above a factory floor. Site A constructs the method, Site B selects policy, and Site C is opened only after the configuration is frozen.

**Success is not an attractive point cloud.** Every quantity must carry a frame and units; projections and transforms must satisfy invariants; outliers must be rejected; relative and metric depth must not be confused; geometric and decision uncertainty must be visible.

The default path is deterministic, credential-free, CPU-safe, and synthetic. Optional remote models are disabled and never execute in this notebook.


### Learning route and safety boundary

You will implement the primitives before reviewing packaged tools:

`frames → projection → back-projection → calibration → epipolar geometry → RANSAC → stereo → triangulation → depth → pose → reconstruction → spatial decision`

This lab performs no network, model download, network operation, or physical action. The final clearance policy is a **demonstration policy**, not a certified metrology or machine-safety standard.

![The course coordinate-frame contract.](assets/coordinate-frames.svg)


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path
import json
import math
import platform
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import least_squares
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation

SEED = 20260912
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
    "mode": "credential_free_synthetic_geometry",
})


## 1. Typed coordinate and camera contracts

Numerical arrays alone cannot prevent a world point from being interpreted as a camera point. These contracts make frame, unit, direction, and transform translation units executable. The canonical camera frame is OpenCV-style: `+x` right, `+y` down, `+z` forward; pixels use `(u, v)` from the top-left. Unit conversion must be explicit at an ingestion boundary.


In [ ]:
@dataclass(frozen=True)
class FramedPoints:
    xyz: np.ndarray
    frame: str
    unit: str = "metre"

    def __post_init__(self):
        values = np.asarray(self.xyz, dtype=float)
        if values.ndim != 2 or values.shape[1] != 3:
            raise ValueError("xyz must have shape [N, 3]")
        if not self.frame or self.unit not in {"metre", "millimetre"}:
            raise ValueError("frame and supported unit are required")
        object.__setattr__(self, "xyz", values)


@dataclass(frozen=True)
class RigidTransform:
    matrix: np.ndarray
    source_frame: str
    target_frame: str
    translation_unit: str = "metre"

    def __post_init__(self):
        matrix = np.asarray(self.matrix, dtype=float)
        if matrix.shape != (4, 4) or not np.isfinite(matrix).all():
            raise ValueError("rigid-transform matrix must be finite and 4x4")
        if not self.source_frame or not self.target_frame:
            raise ValueError("source and target frames are required")
        if self.translation_unit not in {"metre", "millimetre"}:
            raise ValueError("supported translation unit is required")
        if not np.allclose(matrix[3], [0, 0, 0, 1], atol=1e-10):
            raise ValueError("homogeneous rigid transform must end with [0,0,0,1]")
        rotation = matrix[:3, :3]
        if not np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-8) or not np.isclose(np.linalg.det(rotation), 1.0, atol=1e-8):
            raise ValueError("rotation must be orthonormal with determinant +1")
        object.__setattr__(self, "matrix", matrix)


@dataclass(frozen=True)
class CameraModel:
    camera_id: str
    K: np.ndarray
    T_cw: RigidTransform  # maps world coordinates into this camera
    width: int
    height: int
    calibration_version: str

    def __post_init__(self):
        K = np.asarray(self.K, dtype=float)
        if K.shape != (3, 3) or not np.isfinite(K).all():
            raise ValueError("K must be a finite 3x3 matrix")
        if K[0, 0] <= 0 or K[1, 1] <= 0 or not np.isclose(K[2, 2], 1.0):
            raise ValueError("K requires positive focal lengths and K[2,2] = 1")
        if self.T_cw.source_frame != "world" or self.T_cw.target_frame != self.camera_id:
            raise ValueError("T_cw must explicitly map world to camera_id")
        if self.T_cw.translation_unit != "metre":
            raise ValueError("camera extrinsics in this lab must use metres")
        if self.width <= 0 or self.height <= 0:
            raise ValueError("positive image dimensions required")
        object.__setattr__(self, "K", K)


def make_transform(R: np.ndarray, t: np.ndarray, source_frame: str, target_frame: str, translation_unit: str = "metre") -> RigidTransform:
    T = np.eye(4, dtype=float)
    T[:3, :3] = np.asarray(R, dtype=float)
    T[:3, 3] = np.asarray(t, dtype=float)
    return RigidTransform(T, source_frame, target_frame, translation_unit)


def invert_rigid(transform: RigidTransform) -> RigidTransform:
    R_ab, t_ab = transform.matrix[:3, :3], transform.matrix[:3, 3]
    return make_transform(R_ab.T, -R_ab.T @ t_ab, transform.target_frame, transform.source_frame, transform.translation_unit)


def transform_points(points: FramedPoints, transform: RigidTransform) -> FramedPoints:
    if points.frame != transform.source_frame:
        raise ValueError(f"frame mismatch: points are {points.frame}, transform expects {transform.source_frame}")
    if points.unit != transform.translation_unit:
        raise ValueError(f"unit mismatch: points use {points.unit}, transform translation uses {transform.translation_unit}")
    homogeneous = np.c_[points.xyz, np.ones(len(points.xyz))]
    out = (transform.matrix @ homogeneous.T).T[:, :3]
    return FramedPoints(out, transform.target_frame, points.unit)


I = np.eye(4)
T_ab = make_transform(Rotation.from_euler("y", 12, degrees=True).as_matrix(), [0.2, -0.1, 0.3], "frame_b", "frame_a")
assert np.allclose(T_ab.matrix @ invert_rigid(T_ab).matrix, I, atol=1e-10)
assert np.allclose(invert_rigid(invert_rigid(T_ab)).matrix, T_ab.matrix, atol=1e-10)
millimetre_points = FramedPoints(np.array([[1000.0, 0.0, 0.0]]), "frame_b", "millimetre")
try:
    transform_points(millimetre_points, T_ab)
except ValueError as exc:
    assert "unit mismatch" in str(exc)
else:
    raise AssertionError("1000 mm points plus a metre transform must be rejected before computation")
print("Rigid-transform invariants passed; metre/millimetre mismatch rejected before computation.")


### Build a scene and cameras

The world uses metres. The valve cube lies in front of the cameras; the floor is the plane `y = 0.60 m`. For a camera centre $C_w$, world-to-camera translation is $t=-RC_w$—not the camera centre itself.


In [ ]:
def cube_vertices(x0=-0.25, x1=0.25, y0=0.28, y1=0.48, z0=2.45, z1=2.85):
    return np.array([[x, y, z] for x in (x0, x1) for y in (y0, y1) for z in (z0, z1)], float)


K = np.array([[700.0, 0.0, 320.0], [0.0, 700.0, 240.0], [0.0, 0.0, 1.0]])

def camera_from_center(camera_id: str, center_w, yaw_deg=0.0, calibration_version="cal-v1"):
    R_cw = Rotation.from_euler("y", yaw_deg, degrees=True).as_matrix()
    center_w = np.asarray(center_w, float)
    t_cw = -R_cw @ center_w
    return CameraModel(camera_id, K.copy(), make_transform(R_cw, t_cw, "world", camera_id), 640, 480, calibration_version)


cam_left = camera_from_center("camera_left", [0.0, 0.0, 0.0])
cam_right = camera_from_center("camera_right", [0.24, 0.0, 0.0])
cam_oblique = camera_from_center("camera_oblique", [-0.14, -0.02, 0.08], yaw_deg=4.0)
scene = FramedPoints(cube_vertices(), "world")

camera_center = invert_rigid(cam_right.T_cw).matrix[:3, 3]
assert np.allclose(camera_center, [0.24, 0.0, 0.0])
print({"scene_points": len(scene.xyz), "right_camera_center_world_m": camera_center.tolist()})


## 2. Manual pinhole projection

Projection first transforms world points into the camera, then divides x and y by positive camera z, then applies intrinsics. Points behind the camera are invalid observations, not unusual pixels.

![Manual pinhole projection stages.](assets/pinhole-projection.svg)


In [ ]:
def project_points(points_world: FramedPoints, camera: CameraModel) -> tuple[np.ndarray, np.ndarray]:
    if points_world.frame != "world" or points_world.unit != "metre":
        raise ValueError("project_points requires metre-valued world points")
    points_c = transform_points(points_world, camera.T_cw).xyz
    z = points_c[:, 2]
    if np.any(z <= 0):
        raise ValueError("all projected points must have positive camera z")
    normalized = points_c[:, :2] / z[:, None]
    pixels_h = (camera.K @ np.c_[normalized, np.ones(len(normalized))].T).T
    return pixels_h[:, :2], z


axis_point = FramedPoints(np.array([[0.0, 0.0, 2.0]]), "world")
axis_pixel, axis_z = project_points(axis_point, cam_left)
assert np.allclose(axis_pixel[0], [320.0, 240.0])

same_ray = FramedPoints(np.array([[0.2, 0.1, 2.0], [0.4, 0.2, 4.0]]), "world")
same_pixels, _ = project_points(same_ray, cam_left)
assert np.allclose(same_pixels[0], same_pixels[1])

cube_px, cube_z = project_points(scene, cam_left)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cube_px[:, 0], cube_px[:, 1], c=cube_z, s=80, cmap="viridis")
for i, (u, v) in enumerate(cube_px): ax.text(u + 3, v - 3, str(i), fontsize=8)
ax.set(xlim=(0, 640), ylim=(480, 0), xlabel="u (pixel)", ylabel="v (pixel)", title="Cube vertices projected into camera_left")
ax.set_aspect("equal")
plt.show()


## 3. Back-projection: z-depth is not range

$K^{-1}p$ is a ray. Multiplying the unnormalised ray by camera-axis z-depth recovers a camera point. Multiplying a unit ray by Euclidean range uses a different contract.


In [ ]:
def pixel_ray(pixel_uv: np.ndarray, K_matrix: np.ndarray, normalize=True) -> np.ndarray:
    ray = np.linalg.inv(K_matrix) @ np.array([pixel_uv[0], pixel_uv[1], 1.0])
    return ray / np.linalg.norm(ray) if normalize else ray


def backproject_z(pixel_uv: np.ndarray, z_depth_m: float, K_matrix: np.ndarray) -> np.ndarray:
    if z_depth_m <= 0:
        raise ValueError("z-depth must be positive")
    return z_depth_m * pixel_ray(pixel_uv, K_matrix, normalize=False)


pixel = np.array([530.0, 310.0])
z_depth = 3.0
point_from_z = backproject_z(pixel, z_depth, K)
range_m = np.linalg.norm(point_from_z)
point_from_range = range_m * pixel_ray(pixel, K, normalize=True)
assert np.allclose(point_from_z, point_from_range)
assert range_m > z_depth
pd.DataFrame([
    {"quantity": "camera-axis z-depth", "value_m": z_depth},
    {"quantity": "Euclidean range along ray", "value_m": range_m},
])


## 4. Distortion and calibration coverage

The next controlled coverage experiment injects radial distortion. Observations inspected only near the principal point can look healthy because radial effects grow toward the image boundary. We report mean, median, and p95 residuals by region. This is not a fit of weak versus diverse calibration-view sets; the following DLT section is separately a projection-estimation exercise.


In [ ]:
def distort_normalized(xy: np.ndarray, k1=0.0, k2=0.0, p1=0.0, p2=0.0) -> np.ndarray:
    x, y = xy[:, 0], xy[:, 1]
    r2 = x*x + y*y
    radial = 1.0 + k1*r2 + k2*r2*r2
    xd = x*radial + 2*p1*x*y + p2*(r2 + 2*x*x)
    yd = y*radial + p1*(r2 + 2*y*y) + 2*p2*x*y
    return np.c_[xd, yd]


def normalized_to_pixel(xy: np.ndarray, K_matrix: np.ndarray) -> np.ndarray:
    return (K_matrix @ np.c_[xy, np.ones(len(xy))].T).T[:, :2]


grid_u, grid_v = np.meshgrid(np.linspace(20, 620, 15), np.linspace(20, 460, 11))
ideal_px = np.c_[grid_u.ravel(), grid_v.ravel()]
rays = (np.linalg.inv(K) @ np.c_[ideal_px, np.ones(len(ideal_px))].T).T
distorted_px = normalized_to_pixel(distort_normalized(rays[:, :2], k1=-0.22, k2=0.05), K)
residual = np.linalg.norm(distorted_px - ideal_px, axis=1)
radius = np.linalg.norm(ideal_px - np.array([320, 240]), axis=1)
region = np.where(radius < 150, "central weak coverage", "edge coverage")

def error_summary(values):
    values = np.asarray(values)
    return {"mean_px": values.mean(), "median_px": np.median(values), "p95_px": np.quantile(values, 0.95), "n": len(values)}


calibration_residuals = pd.DataFrame([
    {"slice": name, **error_summary(residual[region == name])}
    for name in ["central weak coverage", "edge coverage"]
])
assert calibration_residuals.loc[1, "p95_px"] > calibration_residuals.loc[0, "p95_px"]
calibration_residuals


### Transparent linear camera estimation

DLT estimates a 3×4 projective camera from 3D–2D correspondences. It is useful for exposing the algebra, but production calibration needs normalization, multiple diverse target views, a declared distortion model, nonlinear refinement, and degeneracy checks.


In [ ]:
def estimate_camera_dlt(points_xyz: np.ndarray, pixels_uv: np.ndarray) -> np.ndarray:
    Xh = np.c_[points_xyz, np.ones(len(points_xyz))]
    rows = []
    for X, (u, v) in zip(Xh, pixels_uv):
        rows += [np.r_[X, np.zeros(4), -u*X], np.r_[np.zeros(4), X, -v*X]]
    _, _, vt = np.linalg.svd(np.asarray(rows))
    P = vt[-1].reshape(3, 4)
    return P / P[-1, -1]


def project_matrix(points_xyz: np.ndarray, P: np.ndarray) -> np.ndarray:
    q = (P @ np.c_[points_xyz, np.ones(len(points_xyz))].T).T
    return q[:, :2] / q[:, 2:3]


cal_points = rng.uniform([-0.8, -0.5, 2.0], [0.8, 0.7, 4.5], size=(70, 3))
cal_true, _ = project_points(FramedPoints(cal_points, "world"), cam_oblique)
cal_observed = cal_true + rng.normal(0, 0.25, cal_true.shape)
P_dlt = estimate_camera_dlt(cal_points, cal_observed)
dlt_error = np.linalg.norm(project_matrix(cal_points, P_dlt) - cal_observed, axis=1)
dlt_summary = error_summary(dlt_error)
assert dlt_summary["p95_px"] < 1.0
dlt_summary


## 5. Epipolar geometry and correspondence outliers

For calibrated cameras, relative pose gives $E=[t]_	imes R$, and $F=K_B^{-T}EK_A^{-1}$. A true pair should satisfy $p_B^TFp_A=0$ up to noise.

![Epipolar geometry constrains a match to a line.](assets/epipolar-geometry.svg)


In [ ]:
def skew(v: np.ndarray) -> np.ndarray:
    x, y, z = np.asarray(v, float)
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]], float)


def fundamental_from_cameras(cam_a: CameraModel, cam_b: CameraModel) -> np.ndarray:
    T_ba = cam_b.T_cw.matrix @ invert_rigid(cam_a.T_cw).matrix
    R_ba, t_ba = T_ba[:3, :3], T_ba[:3, 3]
    E = skew(t_ba) @ R_ba
    return np.linalg.inv(cam_b.K).T @ E @ np.linalg.inv(cam_a.K)


def sampson_error(F: np.ndarray, uv_a: np.ndarray, uv_b: np.ndarray) -> np.ndarray:
    a = np.c_[uv_a, np.ones(len(uv_a))]
    b = np.c_[uv_b, np.ones(len(uv_b))]
    Fa = (F @ a.T).T
    Ftb = (F.T @ b.T).T
    numer = np.sum(b * Fa, axis=1) ** 2
    denom = Fa[:, 0]**2 + Fa[:, 1]**2 + Ftb[:, 0]**2 + Ftb[:, 1]**2
    return numer / np.maximum(denom, 1e-12)


match_points = rng.uniform([-0.65, -0.35, 2.0], [0.65, 0.65, 5.0], size=(80, 3))
uv_a, _ = project_points(FramedPoints(match_points, "world"), cam_left)
uv_b, _ = project_points(FramedPoints(match_points, "world"), cam_right)
uv_a += rng.normal(0, 0.25, uv_a.shape)
uv_b += rng.normal(0, 0.25, uv_b.shape)
outlier_ids = rng.choice(len(uv_b), size=16, replace=False)
uv_b_corrupt = uv_b.copy()
uv_b_corrupt[outlier_ids] = uv_b_corrupt[rng.permutation(outlier_ids)]
F_true = fundamental_from_cameras(cam_left, cam_right)
epi_error = np.sqrt(sampson_error(F_true, uv_a, uv_b_corrupt))
is_true_match = np.ones(len(uv_a), bool); is_true_match[outlier_ids] = False
pd.DataFrame([
    {"pair_type": "true correspondences", **error_summary(epi_error[is_true_match])},
    {"pair_type": "injected outliers", **error_summary(epi_error[~is_true_match])},
])


### Normalized eight-point fitting and RANSAC

RANSAC repeatedly proposes geometry from a minimal sample, scores all matches, and keeps the strongest consensus. The random seed, residual definition, threshold, and iteration budget are part of the result.


In [ ]:
def normalize_pixels(uv: np.ndarray):
    mean = uv.mean(axis=0)
    centred = uv - mean
    scale = math.sqrt(2) / max(np.mean(np.linalg.norm(centred, axis=1)), 1e-12)
    T = np.array([[scale, 0, -scale*mean[0]], [0, scale, -scale*mean[1]], [0, 0, 1]])
    homogeneous = (T @ np.c_[uv, np.ones(len(uv))].T).T
    return homogeneous[:, :2], T


def eight_point_F(uv1: np.ndarray, uv2: np.ndarray) -> np.ndarray:
    if len(uv1) < 8:
        raise ValueError("at least eight pairs required")
    a, T1 = normalize_pixels(uv1); b, T2 = normalize_pixels(uv2)
    x1, y1 = a[:, 0], a[:, 1]; x2, y2 = b[:, 0], b[:, 1]
    A = np.c_[x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, np.ones(len(a))]
    _, _, vt = np.linalg.svd(A)
    F = vt[-1].reshape(3, 3)
    u, s, vt2 = np.linalg.svd(F); s[-1] = 0
    F_rank2 = u @ np.diag(s) @ vt2
    F_denorm = T2.T @ F_rank2 @ T1
    return F_denorm / np.linalg.norm(F_denorm)


def ransac_fundamental(uv1, uv2, threshold_px=0.8, iterations=500, seed=SEED):
    local_rng = np.random.default_rng(seed)
    best = None
    for _ in range(iterations):
        ids = local_rng.choice(len(uv1), 8, replace=False)
        F = eight_point_F(uv1[ids], uv2[ids])
        errors = np.sqrt(sampson_error(F, uv1, uv2))
        mask = errors < threshold_px
        score = (int(mask.sum()), -float(np.median(errors[mask])) if mask.any() else -np.inf)
        if best is None or score > best[0]: best = (score, F, mask, errors)
    _, _, mask, _ = best
    refined = eight_point_F(uv1[mask], uv2[mask])
    errors = np.sqrt(sampson_error(refined, uv1, uv2))
    return refined, errors < threshold_px, errors


F_fit, inliers, ransac_errors = ransac_fundamental(uv_a, uv_b_corrupt)
tp = int(np.sum(inliers & is_true_match)); fp = int(np.sum(inliers & ~is_true_match))
fn = int(np.sum(~inliers & is_true_match))
ransac_report = {"inliers": int(inliers.sum()), "precision": tp/max(tp+fp, 1), "recall": tp/max(tp+fn, 1), "median_inlier_px": float(np.median(ransac_errors[inliers]))}
print(ransac_report)
assert ransac_report["precision"] > 0.90 and ransac_report["recall"] > 0.90
ransac_report


## 6. Stereo depth and analytic uncertainty

In rectified stereo, $Z=fB/d$. The next table makes the inverse relationship and derivative visible. The values are model-based estimates, not end-to-end sensor calibration.

![Rectified disparity and depth contract.](assets/stereo-depth.svg)


In [ ]:
def disparity_to_z(disparity_px, focal_px, baseline_m):
    disparity_px = np.asarray(disparity_px, float)
    if np.any(disparity_px <= 0): raise ValueError("positive disparity required")
    return focal_px * baseline_m / disparity_px


def stereo_sigma_z(disparity_px, focal_px, baseline_m, sigma_disparity_px):
    disparity_px = np.asarray(disparity_px, float)
    return focal_px * baseline_m * sigma_disparity_px / disparity_px**2


disparities = np.array([84, 42, 21, 10.5, 5.25])
stereo_table = pd.DataFrame({
    "disparity_px": disparities,
    "z_depth_m": disparity_to_z(disparities, 700, 0.24),
    "sigma_z_m_at_0.35px": stereo_sigma_z(disparities, 700, 0.24, 0.35),
})
assert np.all(np.diff(stereo_table["z_depth_m"]) > 0)
stereo_table


In [ ]:
baseline_rows = []
for baseline in [0.08, 0.16, 0.24, 0.40]:
    for true_z in [2.0, 5.0, 10.0, 20.0]:
        d = 700 * baseline / true_z
        baseline_rows.append({
            "baseline_m": baseline,
            "range_m": true_z,
            "disparity_px": d,
            "sigma_z_m": stereo_sigma_z(d, 700, baseline, 0.35),
            "non_numeric_tradeoff": "occlusion/matching/FOV burden rises with baseline",
        })
baseline_tradeoff = pd.DataFrame(baseline_rows)
fig, ax = plt.subplots(figsize=(7, 4))
for baseline, frame in baseline_tradeoff.groupby("baseline_m"):
    ax.plot(frame.range_m, frame.sigma_z_m, marker="o", label=f"B={baseline:.2f} m")
ax.set(xlabel="true z-depth (m)", ylabel="analytic σz (m)", title="Depth uncertainty grows rapidly with distance")
ax.legend(); ax.grid(alpha=.25); plt.show()


## 7. Triangulation and ray conditioning

DLT triangulation stacks two cross-product projection constraints. It always returns a number unless the linear system is singular; a separate geometric quality check must decide whether that number is useful.

![Strong and weak triangulation geometry.](assets/triangulation-uncertainty.svg)


In [ ]:
def projection_matrix(camera: CameraModel) -> np.ndarray:
    return camera.K @ camera.T_cw.matrix[:3, :]


def triangulate_dlt(uv_a: np.ndarray, uv_b: np.ndarray, cam_a: CameraModel, cam_b: CameraModel) -> np.ndarray:
    P1, P2 = projection_matrix(cam_a), projection_matrix(cam_b)
    points = []
    for (u1, v1), (u2, v2) in zip(uv_a, uv_b):
        A = np.vstack([u1*P1[2]-P1[0], v1*P1[2]-P1[1], u2*P2[2]-P2[0], v2*P2[2]-P2[1]])
        _, _, vt = np.linalg.svd(A)
        X = vt[-1]; points.append(X[:3] / X[3])
    return np.asarray(points)


def ray_angle_deg(point_w, cam_a, cam_b):
    ca = invert_rigid(cam_a.T_cw).matrix[:3, 3]; cb = invert_rigid(cam_b.T_cw).matrix[:3, 3]
    ra = point_w - ca; rb = point_w - cb
    cosine = np.dot(ra, rb) / (np.linalg.norm(ra)*np.linalg.norm(rb))
    return np.degrees(np.arccos(np.clip(cosine, -1, 1)))


tri_points = rng.uniform([-0.5, -0.2, 2.0], [0.5, 0.55, 8.0], size=(120, 3))
tri_a, _ = project_points(FramedPoints(tri_points, "world"), cam_left)
tri_b, _ = project_points(FramedPoints(tri_points, "world"), cam_right)
noisy_a = tri_a + rng.normal(0, .35, tri_a.shape); noisy_b = tri_b + rng.normal(0, .35, tri_b.shape)
tri_est = triangulate_dlt(noisy_a, noisy_b, cam_left, cam_right)
tri_error = np.linalg.norm(tri_est - tri_points, axis=1)
angles = np.array([ray_angle_deg(p, cam_left, cam_right) for p in tri_points])
triangulation_report = pd.DataFrame({"angle_deg": angles, "z_m": tri_points[:, 2], "error_m": tri_error})
triangulation_report.assign(angle_band=pd.cut(angles, bins=[0,2,4,8,90])).groupby("angle_band", observed=True).error_m.agg(["count","median","max"])


### Monte Carlo: the same pixel noise, different geometry

Near-parallel rays amplify the same localisation noise. We repeat the measurement rather than presenting one lucky sample.


In [ ]:
def triangulation_monte_carlo(point_w, cam_a, cam_b, sigma_px=.35, trials=300, seed=0):
    local_rng = np.random.default_rng(seed)
    u1, _ = project_points(FramedPoints(np.asarray(point_w)[None], "world"), cam_a)
    u2, _ = project_points(FramedPoints(np.asarray(point_w)[None], "world"), cam_b)
    estimates = []
    for _ in range(trials):
        estimates.append(triangulate_dlt(u1+local_rng.normal(0,sigma_px,(1,2)), u2+local_rng.normal(0,sigma_px,(1,2)), cam_a, cam_b)[0])
    estimates = np.asarray(estimates)
    return {
        "point": point_w,
        "ray_angle_deg": ray_angle_deg(np.asarray(point_w), cam_a, cam_b),
        "median_error_m": float(np.median(np.linalg.norm(estimates-point_w, axis=1))),
        "p95_error_m": float(np.quantile(np.linalg.norm(estimates-point_w, axis=1), .95)),
        "z_std_m": float(estimates[:,2].std()),
    }


conditioning = pd.DataFrame([
    triangulation_monte_carlo(np.array([0.1, 0.2, 2.0]), cam_left, cam_right, seed=1),
    triangulation_monte_carlo(np.array([0.1, 0.2, 18.0]), cam_left, cam_right, seed=2),
])
assert conditioning.loc[1, "p95_error_m"] > conditioning.loc[0, "p95_error_m"]
conditioning


## 8. Relative versus metric depth

Raw relative depth and metric depth have different evaluation contracts. The raw output receives ordering and scale/shift-invariant shape metrics only. An explicitly oracle-aligned score may use metre-valued ground truth, but it is labelled as shape error after alignment—not zero-shot metric accuracy. The valid-mask policy is explicit and boundary pixels receive a separate slice.


In [ ]:
def align_scale_shift(pred: np.ndarray, target: np.ndarray, valid: np.ndarray) -> tuple[np.ndarray, tuple[float,float]]:
    A = np.c_[pred[valid], np.ones(valid.sum())]
    scale, shift = np.linalg.lstsq(A, target[valid], rcond=None)[0]
    return scale*pred + shift, (float(scale), float(shift))


def relative_depth_metrics(pred, target, valid):
    p, t = pred[valid], target[valid]
    rank_correlation = float(scipy.stats.spearmanr(p, t).statistic)
    stride = 37
    pred_delta, target_delta = p[:-stride] - p[stride:], t[:-stride] - t[stride:]
    comparable = target_delta != 0
    ordering_accuracy = float(np.mean(np.sign(pred_delta[comparable]) == np.sign(target_delta[comparable])))
    aligned, _ = align_scale_shift(pred, target, valid)
    normalized_shape_rmse = float(np.sqrt(np.mean((aligned[valid]-t)**2)) / np.mean(t))
    return {"spearman_rank_correlation": rank_correlation, "pairwise_ordering_accuracy": ordering_accuracy, "scale_shift_invariant_RMSE_normalized": normalized_shape_rmse, "valid_pixels": int(valid.sum())}


def metric_depth_metrics(pred, target, valid):
    p, t = pred[valid], target[valid]
    ratio = np.maximum(p/t, t/p)
    return {"AbsRel": float(np.mean(np.abs(p-t)/t)), "RMSE_m": float(np.sqrt(np.mean((p-t)**2))), "delta_1": float(np.mean(ratio < 1.25)), "valid_pixels": int(valid.sum())}


h, w = 72, 96
yy, xx = np.mgrid[:h, :w]
true_depth = 4.0 + 0.004*xx + 0.006*yy
object_mask = (xx>30)&(xx<68)&(yy>20)&(yy<55)
true_depth[object_mask] -= 1.35
valid_depth = np.ones((h,w), bool); valid_depth[:3] = False; valid_depth[:, :2] = False
relative_output = 0.58*true_depth + 1.7 + rng.normal(0,.025,(h,w))
metric_output = true_depth*1.03 + rng.normal(0,.045,(h,w))
relative_aligned, relative_alignment = align_scale_shift(relative_output, true_depth, valid_depth)

edge = np.zeros_like(object_mask); edge[1:] |= object_mask[1:] != object_mask[:-1]; edge[:,1:] |= object_mask[:,1:] != object_mask[:,:-1]
relative_depth_report = pd.DataFrame([{"prediction": "raw relative output (arbitrary affine units)", **relative_depth_metrics(relative_output, true_depth, valid_depth)}])
oracle_aligned_shape_report = {"prediction": "relative output after oracle scale/shift", "oracle_aligned_shape_RMSE_m": metric_depth_metrics(relative_aligned, true_depth, valid_depth)["RMSE_m"], "claim": "shape evaluation after ground-truth alignment; not metric zero-shot accuracy"}
metric_depth_report = pd.DataFrame([{"prediction": "metric z-depth", **metric_depth_metrics(metric_output, true_depth, valid_depth)}])
edge_report = pd.DataFrame([
    {"prediction": "relative after oracle scale/shift", "evaluation_contract": "oracle_aligned_shape", "edge_RMSE_m": metric_depth_metrics(relative_aligned, true_depth, valid_depth & edge)["RMSE_m"], "metric_zero_shot_claim": False},
    {"prediction": "metric z-depth", "evaluation_contract": "metric_depth", "edge_RMSE_m": metric_depth_metrics(metric_output, true_depth, valid_depth & edge)["RMSE_m"], "metric_zero_shot_claim": True},
])
print("Oracle relative-depth scale/shift fitted for shape evaluation only:", relative_alignment)
display(relative_depth_report, pd.DataFrame([oracle_aligned_shape_report]), metric_depth_report); edge_report


## 9. Pose estimation from 3D–2D correspondences

PnP solves a different problem from triangulation: known 3D points and their image observations constrain a camera pose. We optimise a six-parameter rotation-vector and translation, then report rotation, translation, and reprojection error separately.


In [ ]:
def pose_from_vector(params):
    return Rotation.from_rotvec(params[:3]).as_matrix(), params[3:]


def pose_residual(params, points_w, observed_uv, K_matrix):
    R_cw, t_cw = pose_from_vector(params)
    points_c = (R_cw @ points_w.T).T + t_cw
    if np.any(points_c[:,2] <= .1): return np.full(observed_uv.size, 1e4)
    projected = normalized_to_pixel(points_c[:,:2]/points_c[:,2:3], K_matrix)
    return (projected-observed_uv).ravel()


pose_points = rng.uniform([-.6,-.4,2.2],[.6,.6,4.8],(45,3))
R_pose_true = Rotation.from_euler("xyz", [2.0, -4.0, 1.5], degrees=True).as_matrix()
t_pose_true = np.array([.08, -.03, .14])
cam_pose_true = CameraModel("pose_camera", K, make_transform(R_pose_true,t_pose_true,"world","pose_camera"),640,480,"cal-v1")
pose_uv, _ = project_points(FramedPoints(pose_points,"world"), cam_pose_true)
pose_uv += rng.normal(0,.3,pose_uv.shape)
initial = np.r_[np.zeros(3), [0,0,.05]]
fit = least_squares(pose_residual, initial, args=(pose_points,pose_uv,K), loss="huber", f_scale=1.0, max_nfev=200)
R_fit,t_fit = pose_from_vector(fit.x)
rotation_error = np.degrees(Rotation.from_matrix(R_fit @ R_pose_true.T).magnitude())
translation_error = np.linalg.norm(t_fit-t_pose_true)
pose_summary = {"rotation_error_deg": float(rotation_error), "translation_error_m": float(translation_error), "reprojection_rmse_px": float(np.sqrt(np.mean(pose_residual(fit.x,pose_points,pose_uv,K)**2))), "optimizer_success": bool(fit.success)}
assert rotation_error < 0.2 and translation_error < 0.01
pose_summary


## 10. Multi-view reconstruction and bundle-adjustment objective

Structure from motion estimates camera poses and sparse points, then bundle adjustment refines reprojection consistency. Here cameras remain calibrated and fixed so learners can inspect point refinement without hiding gauge freedom inside a large solver.

![Verified structure-from-motion stages.](assets/sfm-pipeline.svg)


In [ ]:
recon_true = rng.uniform([-.75,-.25,2.1],[.75,.65,5.2],(90,3))
cameras = [cam_left, cam_right, cam_oblique]
observations = []
for camera in cameras:
    uv, _ = project_points(FramedPoints(recon_true,"world"),camera)
    observations.append(uv+rng.normal(0,.45,uv.shape))

initial_points = triangulate_dlt(observations[0],observations[1],cam_left,cam_right)

def point_bundle_residual(flat_points):
    points = flat_points.reshape(-1,3)
    residuals=[]
    for camera, observed in zip(cameras,observations):
        predicted,_ = project_points(FramedPoints(points,"world"),camera)
        residuals.append((predicted-observed).ravel())
    return np.concatenate(residuals)


before_rmse = float(np.sqrt(np.mean(point_bundle_residual(initial_points.ravel())**2)))
refined_fit = least_squares(point_bundle_residual, initial_points.ravel(), loss="huber", f_scale=1.0, max_nfev=80)
refined_points = refined_fit.x.reshape(-1,3)
after_rmse = float(np.sqrt(np.mean(point_bundle_residual(refined_points.ravel())**2)))
reconstruction_improvement = {"before_reprojection_rmse_px":before_rmse,"after_reprojection_rmse_px":after_rmse,"median_3d_error_m":float(np.median(np.linalg.norm(refined_points-recon_true,axis=1)))}
assert after_rmse < before_rmse
reconstruction_improvement


## 11. Reconstruction metrics and density effects

Accuracy asks whether reconstructed points lie near reference geometry; completeness asks whether reference geometry is covered. Here, `symmetric_mean_nn_distance_m` is explicitly the sum of the two directional mean **Euclidean** nearest-neighbour distances. Other sources may call squared, summed, or averaged variants Chamfer distance, so the convention must travel with the value. F-score requires a tolerance with units.


In [ ]:
def cloud_metrics(pred: np.ndarray, truth: np.ndarray, tolerance_m=.03):
    pred_to_true = cKDTree(truth).query(pred)[0]
    true_to_pred = cKDTree(pred).query(truth)[0]
    precision = np.mean(pred_to_true <= tolerance_m)
    recall = np.mean(true_to_pred <= tolerance_m)
    return {
        "accuracy_mean_m": float(pred_to_true.mean()),
        "completeness_mean_m": float(true_to_pred.mean()),
        "symmetric_mean_nn_distance_m": float(pred_to_true.mean()+true_to_pred.mean()),
        "fscore_at_3cm": float(2*precision*recall/max(precision+recall,1e-12)),
        "tolerance_m": tolerance_m,
    }


full_metrics = cloud_metrics(refined_points,recon_true)
sparse_metrics = cloud_metrics(refined_points[::3],recon_true)
pd.DataFrame([{"cloud":"refined full",**full_metrics},{"cloud":"same process, sparse sampling",**sparse_metrics}])


## 12. Point clouds, voxels, and floor clearance

The representation must answer the product question. We voxelise the reconstructed cloud, fit the known floor-like samples with SVD, and compute valve clearance in world metres. Point-cloud density and provenance remain explicit.

![Query-driven 3D representation choices.](assets/representation-landscape.svg)


In [ ]:
def voxelize(points: np.ndarray, voxel_size_m=.10):
    return np.unique(np.floor(points/voxel_size_m).astype(int),axis=0)


def fit_plane_svd(points: np.ndarray):
    centroid=points.mean(axis=0)
    _,_,vt=np.linalg.svd(points-centroid)
    normal=vt[-1]; normal=normal/np.linalg.norm(normal)
    return normal, -float(normal@centroid)


floor_x,floor_z=np.meshgrid(np.linspace(-1.2,1.2,18),np.linspace(1.8,5.2,20))
floor_points=np.c_[floor_x.ravel(), np.full(floor_x.size,.60), floor_z.ravel()]
floor_observed=floor_points+rng.normal(0,[.002,.003,.002],floor_points.shape)
normal,offset=fit_plane_svd(floor_observed)
if normal[1] < 0: normal,offset=-normal,-offset
cube_bottom=scene.xyz[scene.xyz[:,1]==scene.xyz[:,1].max()]
signed_bottom=cube_bottom@normal+offset
clearance_m=float(np.min(np.abs(signed_bottom)))
voxels=voxelize(np.vstack([refined_points,floor_observed]),.10)
print({"occupied_10cm_voxels":len(voxels),"floor_normal_world":normal.tolist(),"clearance_m":clearance_m})

fig=plt.figure(figsize=(8,5)); ax=fig.add_subplot(111,projection="3d")
ax.scatter(floor_observed[:,0],floor_observed[:,2],floor_observed[:,1],s=3,alpha=.25,label="floor observations")
ax.scatter(scene.xyz[:,0],scene.xyz[:,2],scene.xyz[:,1],s=60,label="valve cube")
ax.set(xlabel="world x (m)",ylabel="world z (m)",zlabel="world y/down (m)",title="Frame-aware metric scene")
ax.legend(); plt.show()


## 13. Occlusion is not free space

A z-buffer keeps the nearest positive z-depth when multiple points land in one pixel. Anything behind that visible surface is unobserved from this view, not verified empty space.


In [ ]:
def zbuffer(points_world: FramedPoints, camera: CameraModel):
    uv,z=project_points(points_world,camera)
    ij=np.rint(uv).astype(int)
    valid=(ij[:,0]>=0)&(ij[:,0]<camera.width)&(ij[:,1]>=0)&(ij[:,1]<camera.height)
    depth=np.full((camera.height,camera.width),np.inf)
    owner=np.full((camera.height,camera.width),-1,int)
    for index,((u,v),zz) in enumerate(zip(ij[valid],z[valid])):
        if zz<depth[v,u]: depth[v,u]=zz; owner[v,u]=np.flatnonzero(valid)[index]
    return depth,owner


collinear=FramedPoints(np.array([[0,0,2.0],[0,0,3.5]]),"world")
depth_buffer,owner=zbuffer(collinear,cam_left)
assert np.isclose(depth_buffer[240,320],2.0)
assert owner[240,320]==0
print("Near point is visible; farther collinear point remains occluded/unknown, not free space.")


## 14. Spatial relations and uncertainty-aware policy

Geometry computes metric relations; it should not ask a language model to perform arithmetic. We propagate point/floor noise through Monte Carlo trials and make the review region explicit.

![Spatial evidence from calibrated observations to a reviewable metric decision.](assets/spatial-evidence-pipeline.svg)


In [ ]:
def relation_world(a,b,tolerance=.01):
    delta=np.asarray(a)-np.asarray(b)
    return {
        "a_left_of_b": bool(delta[0] < -tolerance),
        "a_above_b": bool(delta[1] < -tolerance),  # +y is down in this declared world
        "a_in_front_of_b": bool(delta[2] < -tolerance),
        "distance_m": float(np.linalg.norm(delta)),
        "frame":"world",
    }


def clearance_trials(bottom_points, floor_points, sigma_point=.004, trials=1000, seed=17):
    local=np.random.default_rng(seed); values=[]
    for _ in range(trials):
        noisy_floor=floor_points+local.normal(0,sigma_point,floor_points.shape)
        n,d=fit_plane_svd(noisy_floor)
        if n[1]<0:n,d=-n,-d
        noisy_bottom=bottom_points+local.normal(0,sigma_point,bottom_points.shape)
        values.append(np.min(np.abs(noisy_bottom@n+d)))
    return np.asarray(values)


clearance_samples=clearance_trials(cube_bottom,floor_observed)
clearance_interval=np.quantile(clearance_samples,[.025,.5,.975])
DEMONSTRATION_CLEARANCE_LIMIT_M=.115
if clearance_interval[0] >= DEMONSTRATION_CLEARANCE_LIMIT_M: clearance_decision="accept"
elif clearance_interval[2] < DEMONSTRATION_CLEARANCE_LIMIT_M: clearance_decision="reject"
else: clearance_decision="review"
clearance_uncertainty_scope={"interval_label":"teaching_interval_under_point_noise_model","included_sources":["synthetic isotropic point perturbation","floor-plane refit"],"excluded_sources":["calibration covariance","correspondence bias","pose uncertainty","systematic scale error"]}
{"lower_bound_m":clearance_interval[0],"median_m":clearance_interval[1],"upper_bound_m":clearance_interval[2],"limit_m":DEMONSTRATION_CLEARANCE_LIMIT_M,"decision":clearance_decision,**clearance_uncertainty_scope,"notice":"Conditional teaching interval, not a complete 95% coverage claim."}


## 15. Freeze on Site B, report Site C

We now freeze the calibration/matching/reconstruction/clearance policy. Site C changes pixel noise, outlier rate, and calibration bias. No Site C result may change the frozen configuration inside this run.


In [ ]:
FROZEN_POLICY={
    "constructed_on":"Site A",
    "selected_on":"Site B development_only",
    "epipolar_threshold_px":1.2,
    "reconstruction_tolerance_m":.03,
    "clearance_limit_m":DEMONSTRATION_CLEARANCE_LIMIT_M,
    "clearance_interval":"teaching_interval_under_point_noise_model",
    "site_c_role":"reporting_only_no_changes",
    "calibration_version":"cal-v1",
}
POLICY_HASH=sha256(json.dumps(FROZEN_POLICY,sort_keys=True).encode()).hexdigest()
print("Frozen policy hash:",POLICY_HASH)


In [ ]:
def evaluate_source(name, pixel_sigma, outlier_fraction, focal_bias=0.0, seed=0):
    local=np.random.default_rng(seed)
    truth=local.uniform([-.7,-.3,2.0],[.7,.6,7.0],(100,3))
    a,_=project_points(FramedPoints(truth,"world"),cam_left)
    b,_=project_points(FramedPoints(truth,"world"),cam_right)
    a+=local.normal(0,pixel_sigma,a.shape); b+=local.normal(0,pixel_sigma,b.shape)
    n_out=int(len(b)*outlier_fraction); bad=local.choice(len(b),n_out,replace=False)
    b[bad]=b[local.permutation(bad)]
    test_K=K.copy(); test_K[0,0]*=(1+focal_bias); test_K[1,1]*=(1+focal_bias)
    left_test=CameraModel("camera_left",test_K,cam_left.T_cw,640,480,"cal-drift" if focal_bias else "cal-v1")
    right_test=CameraModel("camera_right",test_K,cam_right.T_cw,640,480,"cal-drift" if focal_bias else "cal-v1")
    F=fundamental_from_cameras(left_test,right_test)
    residual=np.sqrt(sampson_error(F,a,b))
    keep=residual<FROZEN_POLICY["epipolar_threshold_px"]
    estimate=triangulate_dlt(a[keep],b[keep],left_test,right_test)
    errors=np.linalg.norm(estimate-truth[keep],axis=1)
    actual_good=np.ones(len(truth),bool);actual_good[bad]=False
    return {
        "source":name,"role":"reporting_only_no_changes" if name=="Site C" else "development_only",
        "pairs":len(truth),"kept":int(keep.sum()),"match_precision":float(np.mean(actual_good[keep])) if keep.any() else 0,
        "required_match_recall":float(np.sum(keep&actual_good)/np.sum(actual_good)),
        "median_epipolar_residual_px":float(np.median(residual[actual_good])),"p95_epipolar_residual_px":float(np.quantile(residual[actual_good],.95)),
        "median_3d_error_m":float(np.median(errors)),"p95_3d_error_m":float(np.quantile(errors,.95)),
        "calibration_version":left_test.calibration_version,"policy_hash":POLICY_HASH,
    }


source_report=pd.DataFrame([
    evaluate_source("Site B",.35,.15,0.0,41),
    evaluate_source("Site C",.75,.24,.012,42),
])
assert source_report.loc[source_report.source=="Site C","policy_hash"].item()==POLICY_HASH
source_report


### Failure attribution, not one score

The Site C report separates epipolar residuals, correspondence retention, and precision from 3D tails and calibration version. In this parallel rig, focal drift can leave epipolar consistency looking plausible while metric 3D error worsens. A release review therefore cannot use epipolar residuals as a calibration certificate, and should inspect range, edge, surface orientation, and decision bands without tuning on the held-out source.


In [ ]:
failure_taxonomy=pd.DataFrame([
    {"failure":"frame_or_unit_mismatch","detector":"transform/unit assertions","outcome":"reject before fusion"},
    {"failure":"calibration_drift","detector":"version mismatch + residual tails","outcome":"quarantine/recalibrate"},
    {"failure":"correspondence_outlier","detector":"epipolar residual","outcome":"reject match"},
    {"failure":"weak_triangulation","detector":"ray angle + propagated error","outcome":"review or restrict range"},
    {"failure":"relative_as_metric","detector":"output contract lacks metric scale","outcome":"block metric claim"},
    {"failure":"occluded_as_free","detector":"visibility state is unknown","outcome":"seek another view"},
    {"failure":"uncertain_clearance","detector":"interval crosses policy limit","outcome":"human review"},
])
failure_taxonomy


## 16. Baselines and representation/tool decision

The options below are not interchangeable. Classical calibrated stereo supplies metric geometry when its assumptions hold; monocular relative depth supplies shape/order; feed-forward multi-view models can jointly predict cameras and geometry but require separate license, memory, uncertainty, and domain review.


In [ ]:
option_matrix=pd.DataFrame([
    {"option":"calibrated stereo + robust geometry","output_contract":"metric z/points","core_default":True,"strength":"traceable scale and constraints","risk":"matching, occlusion, calibration, weak far depth"},
    {"option":"monocular relative-depth model","output_contract":"relative depth","core_default":False,"strength":"dense single-view structure","risk":"no metric claim without scale evidence"},
    {"option":"monocular metric-depth model","output_contract":"model-specific metric depth/points","core_default":False,"strength":"single-view metric prior","risk":"camera/domain/uncertainty calibration"},
    {"option":"optimisation-based SfM/MVS","output_contract":"camera poses + sparse/dense geometry up to gauge","core_default":False,"strength":"inspectable multi-view optimisation","risk":"feature/view-graph and compute failures"},
    {"option":"feed-forward VGGT-family reconstruction","output_contract":"model-specific cameras/depth/points/tracks","core_default":False,"strength":"fast unified prediction","risk":"GPU memory, license, opaque/domain failures"},
])
option_matrix


## 17. Optional integrations — disabled and isolated

These constants are source-governance records, not executable installation instructions. Re-resolve dependencies in separate environments. Verify checkpoint hashes and licenses independently of code licenses.


In [ ]:
CV_ENABLE_DEPTH_ANYTHING_V2=False
CV_ENABLE_UNIDEPTH=False
CV_ENABLE_VGGT=False
CV_ENABLE_VGGT_OMEGA=False
CV_ENABLE_OPEN3D=False

OPTIONAL_SOURCES={
    "Depth Anything V2":{"repo":"https://github.com/DepthAnything/Depth-Anything-V2","revision":"a561b849ebae10a6f5ef49e26c83cbbcd36c71bf","contract":"relative by default; separate metric variants","trust_remote_code":False},
    "UniDepth":{"repo":"https://github.com/lpiccinelli-eth/UniDepth","revision":"8d8cfe4c7ee15297099983607febf0d4f32eb3d6","contract":"metric depth/point prediction","trust_remote_code":False},
    "VGGT":{"repo":"https://github.com/facebookresearch/vggt","revision":"a288dd0f14786c93483e45524328726ab7b1b4ce","contract":"cameras, depth, point maps, tracks","trust_remote_code":False},
    "VGGT-Omega":{"repo":"https://github.com/facebookresearch/vggt-omega","revision":"b2c61f6631d9f344a2d914bfba5d9529d6fc1d35","contract":"static/dynamic feed-forward reconstruction","trust_remote_code":False,"license_note":"FAIR non-commercial research license at reviewed revision"},
    "COLMAP":{"repo":"https://github.com/colmap/colmap","revision":"d3ccaf358e00936db2bd290f2623a652b00e80bb","contract":"optimisation-based SfM/MVS"},
    "Open3D":{"repo":"https://github.com/isl-org/Open3D","revision":"1a9eb990f9a20936c30c428568c602bdef760744","contract":"3D data processing and visualization"},
}
assert not any([CV_ENABLE_DEPTH_ANYTHING_V2,CV_ENABLE_UNIDEPTH,CV_ENABLE_VGGT,CV_ENABLE_VGGT_OMEGA,CV_ENABLE_OPEN3D])
pd.DataFrame(OPTIONAL_SOURCES).T[["revision","contract","trust_remote_code"]].fillna("n/a")


## 18. Save a governed evidence artifact

The artifact records observed metrics, the frozen policy hash, frame/unit conventions, calibration version, optional source revisions, and limitations. It does not serialize notebook state as unquestioned truth.


In [ ]:
artifact={
    "course":"Advanced 01 — 3D Vision & Spatial Intelligence",
    "scenario":"synthetic stereo factory metrology",
    "seed":SEED,
    "coordinate_contract":{"world":"x right, y down, z forward","camera":"OpenCV-style +z forward","pixel":"u right, v down","unit":"metre"},
    "policy":FROZEN_POLICY,"policy_hash":POLICY_HASH,
    "calibration_residuals":calibration_residuals.to_dict(orient="records"),
    "ransac":ransac_report,"pose":pose_summary,"reconstruction":reconstruction_improvement,
    "source_report":source_report.to_dict(orient="records"),
    "depth_evaluation":{"raw_relative":relative_depth_report.to_dict(orient="records"),"oracle_aligned_shape":oracle_aligned_shape_report,"metric":metric_depth_report.to_dict(orient="records")},
    "clearance":{"samples":len(clearance_samples),"teaching_interval_under_point_noise_model_m":clearance_interval.tolist(),"decision":clearance_decision,"uncertainty_scope":clearance_uncertainty_scope},
    "optional_sources":OPTIONAL_SOURCES,
    "limitations":["synthetic geometry","no real sensor synchronisation","no model checkpoints executed","demonstration thresholds only","Site C is one controlled shift, not external validity"],
}
artifact_dir=Path("artifacts");artifact_dir.mkdir(exist_ok=True)
(artifact_dir/"spatial_intelligence_evidence.json").write_text(json.dumps(artifact,indent=2),encoding="utf-8")
option_matrix.to_csv(artifact_dir/"spatial_tool_decision.csv",index=False)
print({"artifact":str(artifact_dir/"spatial_intelligence_evidence.json"),"policy_hash":POLICY_HASH,"site_c_role":"reporting_only_no_changes"})


## 19. Production upgrade map

| Teaching lab | Production requirement |
| --- | --- |
| synthetic cameras | immutable calibration registry; focus/resolution state; health monitoring |
| paired observations | hardware timestamps, synchronization uncertainty, dropped-frame policy |
| NumPy projection and DLT | tested OpenCV/COLMAP kernels plus parity fixtures and degeneracy handling |
| fixed cameras in refinement | robust joint bundle adjustment with gauge anchors and covariance/quality evidence |
| one point cloud | versioned spatial store with frame graph, visibility, lifecycle, and access controls |
| Monte Carlo clearance | validated measurement-system uncertainty and hazard-derived limits |
| optional model constants | isolated runtime, checkpoint hashes, license approval, GPU profiling, domain eval |
| one Site C shift | multiple independent sites, devices, seasons, operators, and incident replay |

Operational dashboards should separate calibration drift, correspondence failure, pose failure, reconstruction quality, uncertainty coverage, decision review rate, latency, and hardware health.


## 20. Exercises and explain-without-code checkpoint

### Exercises

1. Add tangential distortion and an iterative undistortion routine; test a round trip.
2. Add an explicit millimetre-to-metre conversion boundary and prove implicit mixing remains rejected.
3. Compare stereo uncertainty across focal length, baseline, range, and disparity noise.
4. Add surface-normal error beside symmetric mean nearest-neighbour distance and F-score.
5. Replace fixed-camera point refinement with a gauge-anchored small joint optimisation.
6. Design a calibration registry and rollback record for two sensor modes.

### Explain without code

- Why does one pixel identify a ray rather than a unique point?
- Why is extrinsic translation not normally the camera centre?
- How can low mean calibration error hide unsafe edge geometry?
- Why does similarity matching need geometric verification?
- Why does far stereo depth amplify the same disparity error?
- Why can a visually convincing reconstruction be metrically wrong?
- Why is occluded space unknown rather than empty?
- What evidence must support an 8 cm physical-clearance claim?

**Next:** Advanced 02 will build on these camera, pose, geometry, and evaluation contracts to study neural rendering and Gaussian scene representations.
